In [90]:
import pandas as pd
df = pd.read_csv('../data/raw/tech_layoffs_til_2025.csv')

Standardize "United Kingdom" to "UK" since UK is the more common spelling 
in this dataset (74 vs 5 occurrences).

In [91]:
df['Country'] = df['Country'].replace('United Kingdom', 'UK')

Industry column has 118 categories; many appear only 1-2 times. 
Grouping low-frequency categories (appearing fewer than 5 times) into 
"Other" to make industry-level analysis more meaningful.

In [92]:
industry_counts = df['Industry'].value_counts()
rare_industries = industry_counts[industry_counts < 5].index
df['Industry'] = df['Industry'].apply(lambda x: 'Other' if x in rare_industries else x)

In [93]:
df['Industry'].value_counts()

Industry
Finance                          289
Other                            228
Retail                           176
Healthcare                       156
Transportation                   142
Consumer                         134
Food                             133
Marketing                        121
HR                                79
Security                          79
Education                         77
Real Estate                       74
Media                             74
Data                              66
Crypto                            65
Travel                            52
Software Development              44
Logistics                         44
Sales                             41
Infrastructure                    37
Energy                            32
Product                           30
Hardware                          29
Support                           29
Fitness                           22
AI                                17
Gaming                       

"Unknown" in the Stage column (356 records) carries no usable funding-stage 
information. Converting it to a proper missing value (NaN) so it's handled 
consistently with other missing data during analysis.

In [94]:
import numpy as np
df['Stage'] = df['Stage'].replace('Unknown', np.nan)

In [95]:
# To verify if there's still 'Unknown' in 'Stage' column
df['Stage'].value_counts()

Stage
Post-IPO          574
Series B          266
Series C          211
Series D          203
Acquired          176
Series E          123
Series A          116
Series F           66
Seed               49
Private Equity     37
Series H           27
Series G           19
Subsidiary         11
Series I            8
Series J            6
Name: count, dtype: int64

Region column is too coarse for analysis (67% classified as "other") and 
is being dropped. Country will be used instead for geographic analysis.

In [96]:
df = df.drop(columns = ['Region'])

Binning Company_Size_before_Layoffs into 3 categories based on the quantile 
analysis from the exploration phase: Small (≤300), Mid-size (301-1000), 
Large (>1000).

In [97]:
df['Company_Size_Category'] = pd.cut(
    df['Company_Size_before_Layoffs'],
    bins=[0, 300, 1000, 99999999999],
    labels=['Small', 'Mid-size', 'Large']
)

In [98]:
df['Company_Size_Category'].value_counts()

Company_Size_Category
Small       587
Mid-size    585
Large       584
Name: count, dtype: int64

Dropping `Nr` (a row index with no analytical value) and `Continent` 
(too coarse, overlaps with Country which is already used for geographic 
analysis).

In [99]:
df = df.drop(columns=['Nr', 'Continent'])

In [100]:
df.to_csv('../data/cleaned/tech_layoffs_cleaned.csv', index=False)

In [101]:
df.columns

Index(['Company', 'Location_HQ', 'USState', 'Country', 'Laid_Off',
       'Date_layoffs', 'Percentage', 'Company_Size_before_Layoffs',
       'Company_Size_after_layoffs', 'Industry', 'Stage',
       'Money_Raised_in__mil', 'Year', 'latitude', 'longitude',
       'Company_Size_Category'],
      dtype='object')